In [1]:
import pandas as pd

# Load dataset
transactional_data = pd.read_excel("./data/Online Retail.xlsx")

# First 5 rows
print(transactional_data.head())



  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  


In [2]:

# data types

print(transactional_data.dtypes)

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object


In [3]:
# missing values 

print(transactional_data.isna().sum())

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


In [4]:
# Duplicates
print("Duplicates:", transactional_data.duplicated().sum())

Duplicates: 5268


In [5]:
# Descriptive Statistics

print(transactional_data.describe(include="all"))

        InvoiceNo StockCode                         Description  \
count    541909.0    541909                              540455   
unique    25900.0      4070                                4223   
top      573585.0    85123A  WHITE HANGING HEART T-LIGHT HOLDER   
freq       1114.0      2313                                2369   
mean          NaN       NaN                                 NaN   
min           NaN       NaN                                 NaN   
25%           NaN       NaN                                 NaN   
50%           NaN       NaN                                 NaN   
75%           NaN       NaN                                 NaN   
max           NaN       NaN                                 NaN   
std           NaN       NaN                                 NaN   

             Quantity                 InvoiceDate      UnitPrice  \
count   541909.000000                      541909  541909.000000   
unique            NaN                         NaN          

In [6]:
# create the copy of Data 

retail_clean = transactional_data.copy()

print("Retail Clean : \n" , retail_clean.head())




Retail Clean : 
   InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  


In [7]:
print("Before duplicates cleaning ")

print("Duplicates : " , retail_clean.duplicated().sum())

retail_clean.drop_duplicates(inplace=True)

print("After duplicates cleaning ")

print("Duplicates : " , retail_clean.duplicated().sum())

Before duplicates cleaning 


Duplicates :  5268


After duplicates cleaning 


Duplicates :  0


In [8]:
# Country
print(retail_clean["Country"].value_counts())

# StockCode
print(retail_clean["StockCode"].sample(20))
print(retail_clean["StockCode"].str.strip().equals(retail_clean["StockCode"]))

# Description
print(retail_clean["Description"].sample(20))
print(retail_clean["Description"].str.strip().equals(retail_clean["Description"]))
print((retail_clean["Description"].str.upper() == retail_clean["Description"]).all())

Country
United Kingdom          490300
Germany                   9480
France                    8541
EIRE                      8184
Spain                     2528
Netherlands               2371
Belgium                   2069
Switzerland               1994
Portugal                  1510
Australia                 1258
Norway                    1086
Italy                      803
Channel Islands            757
Finland                    695
Cyprus                     611
Sweden                     461
Unspecified                442
Austria                    401
Denmark                    389
Japan                      358
Poland                     341
Israel                     294
USA                        291
Hong Kong                  284
Singapore                  229
Iceland                    182
Canada                     151
Greece                     146
Malta                      127
United Arab Emirates        68
European Community          61
RSA                         58


False
146278         HANGING HEN ON NEST DECORATION
111810     DOORMAT FANCY FONT HOME SWEET HOME
437659          SILVER BELLS TABLE DECORATION
311382             VINTAGE UNION JACK BUNTING
258725              DAIRY MAID LARGE MILK JUG
117311          DOORMAT HOME SWEET HOME BLUE 
322183                 36 PENCILS TUBE SKULLS
318987           HOUSE SHAPE PENCIL SHARPENER
261769     WHITE HANGING HEART T-LIGHT HOLDER
231512               REGENCY CAKESTAND 3 TIER
396503     SET/6 TURQUOISE BUTTERFLY T-LIGHTS
264794        NATURAL SLATE HEART CHALKBOARD 
438715      HOT WATER BOTTLE TEA AND SYMPATHY
507984        RECYCLED ACAPULCO MAT TURQUOISE
335076           REX CASH+CARRY JUMBO SHOPPER
73854          72 SWEETHEART FAIRY CAKE CASES
239572                    PINK POLKADOT WRAP 
268249           RED RETROSPOT SMALL MILK JUG
38218     CHARLIE + LOLA RED HOT WATER BOTTLE
37773         HOT WATER BOTTLE I AM SO POORLY
Name: Description, dtype: object


False


False


In [9]:
# Data Cleaning for Outlier Detection
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

# Remove cancelled orders and negative prices
retail_clean = retail_clean[(retail_clean['Quantity'] > 0) & (retail_clean['UnitPrice'] > 0)]
# Drop rows with missing CustomerID to ensure clean continuous features
retail_clean = retail_clean.dropna(subset=['CustomerID'])

# We'll focus on 'Quantity' and 'UnitPrice' for outlier detection.
# To keep notebook execution fast and efficient for KNN, we sample 10,000 records.
df_sample = retail_clean.sample(10000, random_state=42).copy()
print("Sample size for Outlier Detection:", df_sample.shape)


Sample size for Outlier Detection: (10000, 8)


In [10]:
# 1. Statistical Approach: IQR (Interquartile Range)
Q1 = df_sample[['Quantity', 'UnitPrice']].quantile(0.25)
Q3 = df_sample[['Quantity', 'UnitPrice']].quantile(0.75)
IQR = Q3 - Q1

# Define bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Identify outliers
iqr_outliers = df_sample[((df_sample[['Quantity', 'UnitPrice']] < lower_bound) | (df_sample[['Quantity', 'UnitPrice']] > upper_bound)).any(axis=1)]
print(f"Number of IQR Outliers: {len(iqr_outliers)}")
print(iqr_outliers[['Quantity', 'UnitPrice']].head())


Number of IQR Outliers: 1550
        Quantity  UnitPrice
107320         4      12.75
491767         5       8.25
444738        96       1.25
537305         2       8.50
38623         12       8.50


In [11]:
# 2. Statistical Approach: Z-Score
# stats.zscore returns a numpy array or dataframe depending on pandas version. 
# We'll apply it directly to the columns and create a new dataframe
z_scores = np.abs(stats.zscore(df_sample[['Quantity', 'UnitPrice']]))

# Threshold usually set to 3
threshold = 3
z_outliers = df_sample[(z_scores > threshold).any(axis=1)]
print(f"Number of Z-Score Outliers: {len(z_outliers)}")
print(z_outliers[['Quantity', 'UnitPrice']].head())


Number of Z-Score Outliers: 226
        Quantity  UnitPrice
144086       144       1.45
215450         5      18.00
373674       120       3.29
295645       120       0.42
452573         3      18.00


In [12]:
# 3. Distance-based Approach: KNN
X = df_sample[['Quantity', 'UnitPrice']].values

# Fit KNN (n_neighbors=5 is a common default)
knn = NearestNeighbors(n_neighbors=5)
knn.fit(X)
distances, indices = knn.kneighbors(X)

# The outlier score is the distance to the 5th nearest neighbor
outlier_scores = distances[:, -1]

# Set threshold as the 95th percentile of distances (top 5% furthest points)
knn_threshold = np.percentile(outlier_scores, 95)

df_sample['knn_score'] = outlier_scores
knn_outliers = df_sample[df_sample['knn_score'] > knn_threshold]
print(f"Number of KNN Outliers: {len(knn_outliers)}")
print(knn_outliers[['Quantity', 'UnitPrice', 'knn_score']].head())


Number of KNN Outliers: 462
        Quantity  UnitPrice  knn_score
491767         5       8.25       1.00
38623         12       8.50       0.35
172800        12      11.15       1.35
144086       144       1.45       0.39
363091        60       2.95       0.85
